In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

submission_paths = {
    "data1": ("../input/rerun-seti-e-t-volo-d1-baseline-inference/submission.csv", 0.0),
    "data2": ("../input/seti-bl-spatial-info-tf-tpu/submission.csv", 0.0),
    "data3": ("../input/seti-bl-tf-starter-tpu/submission.csv", 0.0),
    "data4": ("../input/seti-learned-image-resizing/submission.csv", 0.0),
    "data5": ("../input/lb-0-980-efficientnet-b0-more-epoch/submission.csv", 1.0),
    "data6": (
        "../input/inference-5x-ensemble-vanilla-resnet34d-seti/submission.csv",
        0.0,
    ),
}
DEFAULT_ZERO_WEIGHT = 0.0  # keep zero weight for missing external submissions

dfs = {}
for key, (path, weight) in submission_paths.items():
    try:
        dfs[key] = pd.read_csv(path)
    except FileNotFoundError:
        pass



In [2]:
sample_path = os.path.join("..", "input", "sample_submission.csv")
if not os.path.exists(sample_path):
    sample_path = "sample_submission.csv"
base_df = pd.read_csv(sample_path)
assert (
    "id" in base_df.columns and "target" in base_df.columns
), "sample_submission.csv missing required columns"




In [3]:
def extract_features(arr):
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr.astype(np.float32)
    arr = np.clip(arr, -1e6, 1e6)
    mean_val = float(arr.mean())
    std_val = float(arr.std())
    max_val = float(arr.max())
    median_val = float(np.median(arr))
    perc75_val = float(np.percentile(arr, 75))
    return [mean_val, std_val, max_val, median_val, perc75_val]


# ---- ensemble of external submissions ----
ensemble_target = np.zeros(len(base_df), dtype=np.float64)
total_weight = 0.0

for key, df in dfs.items():
    orig_weight = submission_paths[key][1]
    weight = orig_weight if orig_weight > 0.0 else DEFAULT_ZERO_WEIGHT
    merged = base_df[["id"]].merge(
        df[["id", "target"]], on="id", how="left", suffixes=("", "_src")
    )
    mean_sub = merged["target_src"].mean()
    merged["target_src"] = merged["target_src"].fillna(
        mean_sub if not np.isnan(mean_sub) else 0.0
    )
    ensemble_target += weight * merged["target_src"].values
    total_weight += weight

if total_weight == 0.0:
    train_labels_path = os.path.join("..", "input", "train_labels.csv")
    if os.path.exists(train_labels_path):
        train_labels = pd.read_csv(train_labels_path)
        mean_target = train_labels["target"].mean()
    else:
        mean_target = 0.5
    ensemble_target = np.full(len(base_df), mean_target, dtype=np.float64)
else:
    ensemble_target = np.clip(ensemble_target / total_weight, 0.0, 1.0)

# ---- load training data (parallel) ----
train_labels_path = os.path.join("..", "input", "train_labels.csv")
train_labels = pd.read_csv(train_labels_path)

max_train_samples = 40000  # increased from 20000
rng = np.random.default_rng(42)
sampled_ids = rng.choice(
    train_labels["id"].values,
    size=min(max_train_samples, len(train_labels)),
    replace=False,
)

# dictionary for fast target lookup (avoids pandas per‑row search)
id_to_target = dict(zip(train_labels["id"], train_labels["target"]))


def load_feat_target(sid):
    """Load a single .npy file, compute features and fetch the target."""
    folder = sid[0]
    fpath = os.path.join("..", "input", "train", folder, f"{sid}.npy")
    try:
        arr = np.load(fpath)
        feats = extract_features(arr)
        return (sid, feats, id_to_target.get(sid, 0.0))
    except Exception:
        return None


# Thread pool – I/O bound, keep workers modest for reproducibility
from concurrent.futures import ThreadPoolExecutor, as_completed

train_feats = []
train_targets = []

with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_sid = {executor.submit(load_feat_target, sid): sid for sid in sampled_ids}
    for future in as_completed(future_to_sid):
        res = future.result()
        if res is not None:
            _, feats, tgt = res
            train_feats.append(feats)
            train_targets.append(tgt)

if len(train_feats) > 0:
    X_train = np.array(train_feats, dtype=np.float64)
    y_train = np.array(train_targets)
    # Ensure deterministic scikit‑learn behavior
    os.environ["OMP_NUM_THREADS"] = "1"
    logreg = LogisticRegression(
        solver="liblinear", max_iter=1000, class_weight="balanced"
    )
    logreg.fit(X_train, y_train)

    # ---- load test data (parallel) ----
    def load_test_feat(tid):
        folder = tid[0]
        fpath = os.path.join("..", "input", "test", folder, f"{tid}.npy")
        try:
            arr = np.load(fpath)
            return extract_features(arr)
        except Exception:
            return [0.0, 0.0, 0.0, 0.0, 0.0]

    test_feats = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(load_test_feat, tid) for tid in base_df["id"]]
        for fut in as_completed(futures):
            test_feats.append(fut.result())

    X_test = np.array(test_feats, dtype=np.float64)
    model_pred = logreg.predict_proba(X_test)[:, 1]
else:
    model_pred = np.full(len(base_df), 0.5)

model_weight = 1.0  # increased from 0.4
final_pred = (ensemble_target * total_weight + model_pred * model_weight) / (
    total_weight + model_weight
)
final_pred = np.clip(final_pred, 0.0, 1.0)

base_df["target"] = final_pred



In [4]:
output_path = "submission.csv"
base_df.to_csv(output_path, index=False)
print(f"Submission written to {output_path} with {len(base_df)} rows.")

Submission written to submission.csv with 6000 rows.
